# Project: Investigate a Dataset - Medical Appointment No Shows

## Table of Contents
<ul>
<li><a href="#intro">Introduction</a></li>
<li><a href="#wrangling">Data Wrangling</a></li>
<li><a href="#eda">Exploratory Data Analysis</a></li>
<li><a href="#conclusions">Conclusions</a></li>
</ul>

<a id='intro'></a>
## Introduction

<a id='dataset'></a>
### Dataset Description 

We will be analyzing a dataset containing information about medical appointments made between Nov 10, 2015 and June 8, 2016 in the city of Vitória, Brazil, with the aim of investigating factors that may correlate with missed appointments.

The dataset was sourced from [Kaggle: Medical Appointment No Shows](https://www.kaggle.com/datasets/joniarroba/noshowappointments). It contains data on 100K+ medical appointments with information about each appointment and the patient. For this analysis, some columns have been renamed for clarity.

| column name | description | original name |
|---|---|---|
| `PatientId` | unique identifier for each patient |`PatientId`|
| `AppointmentId` | unique identifier for each appointment |`AppointmentID`|
| `Gender` | Male or Female |`Gender`|
| `SchedulingDate` | date the appointment was created - [reference](https://www.kaggle.com/datasets/joniarroba/noshowappointments/discussion/340475) |`ScheduledDay`|
| `AppointmentDate` | date of the appointment - [reference](https://www.kaggle.com/datasets/joniarroba/noshowappointments/discussion/340475) |`AppointmentDay`|
| `Age` | age of the patient |`Age`|
| `Neighborhood` | neighborhood of Vitória, Brazil where the appointment takes place |`Neighbourhood`|
| `FinancialAssistance` | patient enrolled in [Bolsa Família](https://en.wikipedia.org/wiki/Bolsa_Fam%C3%ADlia) financial assistance program |`Scholarship`|
| `Hypertension` | patient diagnosed with hypertension |`Hipertension`|
| `Diabetes` | patient diagnosed with diabetes |`Diabetes`|
| `Alcoholism` | patient diagnosed with alcoholism |`Alcoholism`|
| `NumHandicaps` | number of handicaps - [reference](https://www.kaggle.com/datasets/joniarroba/noshowappointments/discussion/29699#229356) |`Handcap`|
| `SMS_received` | 1 or more text messages were sent to the patient |`SMS_received`|
| `NoShow` | the patient missed the appointment |`No-show`|

### Expanded Dataset
Additional columns have been added to the dataset, expanded from existing columns
| column name | description | derived from |
| --- | --- | --- |
| `SchedulingDate_dayName` | day of week the appointment was created | `SchedulingDate` |
| `AppointmentDate_dayName` | day of week of the appointment | `AppointmentDate` |
| `numDaysCreatedToScheduled` | number of days elapsed between appointment creation and appointment | `SchedulingDate` and `AppointmentDate` |


### Question(s) for Analysis
- Do any of a patient's demographic or health factors correlate to a higher likelihood of missing an appointment?
- Do any factors related to the appointment itself correlate to a higher likelihood of a missed appointment?

<a id='wrangling'></a>
## Data Wrangling

### Environment Setup

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm;
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

%config InlineBackend.figure_format = 'svg'
%matplotlib inline

### General Properties
- 110527 records found
- no empty fields found

In [2]:
df = pd.read_csv(
    "./Database_No_show_appointments/noshowappointments-kagglev2-may-2016.csv"
)

df = df.convert_dtypes()

print(f"rows: {df.shape[0]}")
print(f"any na?: {df.isna().sum().any()}")
print(f"any null?: {df.isnull().sum().any()}\n")

print(df.info())

rows: 110527
any na?: False
any null?: False

<class 'pandas.DataFrame'>
RangeIndex: 110527 entries, 0 to 110526
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   PatientId       110527 non-null  Float64
 1   AppointmentID   110527 non-null  Int64  
 2   Gender          110527 non-null  string 
 3   ScheduledDay    110527 non-null  string 
 4   AppointmentDay  110527 non-null  string 
 5   Age             110527 non-null  Int64  
 6   Neighbourhood   110527 non-null  string 
 7   Scholarship     110527 non-null  Int64  
 8   Hipertension    110527 non-null  Int64  
 9   Diabetes        110527 non-null  Int64  
 10  Alcoholism      110527 non-null  Int64  
 11  Handcap         110527 non-null  Int64  
 12  SMS_received    110527 non-null  Int64  
 13  No-show         110527 non-null  string 
dtypes: Float64(1), Int64(8), string(5)
memory usage: 12.8 MB
None


### Data Cleaning
#### Column naming
Renaming columns for clarity - [see Dataset Description](#dataset) for more info
- `AppointmentID` --> `AppointmentId` for consistency
- `ScheduledDay` --> `SchedulingDate` to better reflect data type
- `AppointmentDay` --> `AppointmentDate` for clarity and to reflect data type
- `Neighbourhood` --> `Neigborhood` (US English spelling)
- `Scholarship` --> `FinancialAssistance` for clarity
- `Hipertension` --> `Hypertension` (US English spelling)
- `Handcap` --> `NumHandicaps` for clarity
- `No-show` --> `NoShow` for consitency and Python compatibility

In [3]:
df.rename(
    columns={
        "AppointmentID": "AppointmentId",
        "ScheduledDay": "SchedulingDate",
        "AppointmentDay": "AppointmentDate",
        "Neighbourhood": "Neighborhood",
        "Scholarship": "FinancialAssistance",
        "Hipertension": "Hypertension",
        "Handcap": "NumHandicaps",
        "No-show": "NoShow"
    },
    inplace=True,
)

#### PatientId

There are many appointments with recurring `PatientId` values.

Within each duplicate `PatientId`, there are no observed differences across:
- `Gender`
- `Hypertension`
- `Diabetes`
- `Alcoholism`
- `numHandicaps`
- `Neighborhood`
- `FinancialAssistance`

The dataset will be examined in terms of individual appointments and individual patients, therefore rows with duplicate `PatientId`s will remain.

In [4]:
print(f"unique PatientIds: {df['PatientId'].nunique()}")
print(f"duplicated PatientIds: {df['PatientId'].duplicated().sum()}")
print(f"duplicate PatientIds incl other factors: {df.duplicated(subset=[
        "PatientId",
        "Gender",
        "Hypertension",
        "Diabetes",
        "Alcoholism",
        "NumHandicaps",
        "Neighborhood",
        "FinancialAssistance"
        ]).sum()}")

unique PatientIds: 62299
duplicated PatientIds: 48228
duplicate PatientIds incl other factors: 48228


#### AppointmentId
- no duplicate `AppointmentId`s found
- `AppointmentId` will be used as the DataFrame index

In [5]:
print(f"duplicate AppointmentIds: {df['AppointmentId'].duplicated().sum()}\n")
df.set_index("AppointmentId", inplace=True)

duplicate AppointmentIds: 0



#### Gender
- 2 values found in data: `["F", "M"]` 
- `Gender` converted to categorical datatype `["female", "male"]`

In [6]:
print(df["Gender"].unique())

df["Gender"] = df["Gender"].map({"F": "female", "M": "male"})
gender_categories = df["Gender"].unique()

gender_category = pd.api.types.CategoricalDtype(
    ordered=False, categories=gender_categories)

df["Gender"] = df["Gender"].astype(gender_category)
print()
print(df["Gender"].unique())

<StringArray>
['F', 'M']
Length: 2, dtype: string

['female', 'male']
Categories (2, str): ['female', 'male']


#### SchedulingDate and AppointmentDate
- `SchedulingDate` and `AppointmentDate` converted to `datetime` datatype
- timestamps are GMT
- time of day not recorded for `AppointmentDate`
- time of day was recorded for `SchedulingDate`, but this is not helpful information. Will zero out time of day to enable like-for-like comparison to `AppointmentDate`
- dropping rows where `AppointmentDate` is recorded as occuring before `SchedulingDate` 

In [7]:
df["SchedulingDate"] = pd.to_datetime(df["SchedulingDate"]).dt.normalize()
df["AppointmentDate"] = pd.to_datetime(df["AppointmentDate"])

before = df.shape[0]
df = df[df["AppointmentDate"].dt.date >= df["SchedulingDate"].dt.date]
after = df.shape[0]

print(f"dropped {before - after} row(s) with bad timestamps")

# SchedulingDate - day of week
df["SchedulingDate_dayName"] = df["SchedulingDate"].dt.day_name()

# AppointmentDate - day of week
df["AppointmentDate_dayName"] = df["AppointmentDate"].dt.day_name()

# number of days: SchedulingDate -> AppointmentDate 
df["numDaysCreatedToScheduled"] = (
    df["AppointmentDate"] - df["SchedulingDate"]
).dt.days


dropped 5 row(s) with bad timestamps


#### Age
- Droped rows with invalid Age values `Age < 0`
- `Age == 0` assumed to be an infant
- Right-sized data type to cover realistic range of ages

In [8]:
df["Age"] = df["Age"].astype("Int8")

before = df.shape[0]
df = df[df["Age"] >= 0]
after = df.shape[0]

print(f"dropped {before - after} row(s) with invalid ages")

dropped 1 row(s) with invalid ages


#### Neighborhood

- 81 unique neighborhoods
- converted to categorical datatype

In [9]:
print(f"unique neighborhoods: {df['Neighborhood'].nunique()}")
neighborhood_categories = df["Neighborhood"].unique()

neighborhood_category = pd.api.types.CategoricalDtype(
    ordered=False, categories=neighborhood_categories)

df["Neighborhood"] = df["Neighborhood"].astype(neighborhood_category)

unique neighborhoods: 81


##### Encoded Booleans
Converted binary columns (`[0,1]`, `["Yes","No"]`) to boolean
- `FinancialAssistance`,
- `Hypertension`,
- `Diabetes`,
- `Alcoholism`,
- `SMS_received`,
- `NoShow`

In [10]:
binaryColumns = [
    "FinancialAssistance",
    "Hypertension",
    "Diabetes",
    "Alcoholism",
    "SMS_received",
    "NoShow",
]

for colName in binaryColumns:
    print(f"{df[colName].value_counts()}\n")

df["FinancialAssistance"] = df["FinancialAssistance"].astype("boolean")
df["Hypertension"] = df["Hypertension"].astype("boolean")
df["Diabetes"] = df["Diabetes"].astype("boolean")
df["Alcoholism"] = df["Alcoholism"].astype("boolean")
df["SMS_received"] = df["SMS_received"].astype("boolean")
df["NoShow"] = df["NoShow"].map({"Yes": True, "No": False}).astype("boolean")

FinancialAssistance
0    99660
1    10861
Name: count, dtype: Int64

Hypertension
0    88720
1    21801
Name: count, dtype: Int64

Diabetes
0    102578
1      7943
Name: count, dtype: Int64

Alcoholism
0    107161
1      3360
Name: count, dtype: Int64

SMS_received
0    75039
1    35482
Name: count, dtype: Int64

NoShow
No     88207
Yes    22314
Name: count, dtype: Int64



#### numHandcap
- Right-sized datatype to fit value range

In [11]:
print(df["NumHandicaps"].value_counts())
df["NumHandicaps"] = df["NumHandicaps"].astype("Int8")

NumHandicaps
0    108282
1      2040
2       183
3        13
4         3
Name: count, dtype: Int64


#### Data Cleanup results

In [12]:
df.info()

<class 'pandas.DataFrame'>
Index: 110521 entries, 5642903 to 5629448
Data columns (total 16 columns):
 #   Column                     Non-Null Count   Dtype              
---  ------                     --------------   -----              
 0   PatientId                  110521 non-null  Float64            
 1   Gender                     110521 non-null  category           
 2   SchedulingDate             110521 non-null  datetime64[us, UTC]
 3   AppointmentDate            110521 non-null  datetime64[us, UTC]
 4   Age                        110521 non-null  Int8               
 5   Neighborhood               110521 non-null  category           
 6   FinancialAssistance        110521 non-null  boolean            
 7   Hypertension               110521 non-null  boolean            
 8   Diabetes                   110521 non-null  boolean            
 9   Alcoholism                 110521 non-null  boolean            
 10  NumHandicaps               110521 non-null  Int8               
 

In [13]:
df.head()

,PatientId,Gender,SchedulingDate,AppointmentDate,Age,Neighborhood,FinancialAssistance,Hypertension,Diabetes,Alcoholism,NumHandicaps,SMS_received,NoShow,SchedulingDate_dayName,AppointmentDate_dayName,numDaysCreatedToScheduled
AppointmentId,,,,,,,,,,,,,,,,
5642903,29872499824296.0,female,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,62,JARDIM DA PENHA,False,True,False,False,0,False,False,Friday,Friday,0
5642503,558997776694438.0,male,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,False,False,False,False,0,False,False,Friday,Friday,0
5642549,4262962299951.0,female,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,62,MATA DA PRAIA,False,False,False,False,0,False,False,Friday,Friday,0
5642828,867951213174.0,female,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,8,PONTAL DE CAMBURI,False,False,False,False,0,False,False,Friday,Friday,0
5642494,8841186448183.0,female,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,False,True,True,False,0,False,False,Friday,Friday,0


<a id='eda'></a>
## Exploratory Data Analysis

> **Tip**: Now that you've trimmed and cleaned your data, you're ready to move on to exploration. **Compute statistics** and **create visualizations** with the goal of addressing the research questions that you posed in the Introduction section. You should compute the relevant statistics throughout the analysis when an inference is made about the data. Note that at least two or more kinds of plots should be created as part of the exploration, and you must  compare and show trends in the varied visualizations. Remember to utilize the visualizations that the pandas library already has available.



> **Tip**: Investigate the stated question(s) from multiple angles. It is recommended that you be systematic with your approach. Look at one variable at a time, and then follow it up by looking at relationships between variables. You should explore at least three variables in relation to the primary question. This can be an exploratory relationship between three variables of interest, or looking at how two independent variables relate to a single dependent variable of interest. Lastly, you  should perform both single-variable (1d) and multiple-variable (2d) explorations.


### Do any of a patient's demographic or health factors correlate to a higher likelihood of a missed appointment?

#### Patient demographic and health factors
- `Gender`
- `Age`
- `Neighborhood`
- `FinancialAssistance`
- `Hypertension`
- `Diabetes`
- `Alcoholism`
- `NumHandicaps`

#### Distribution of each factor in the dataset


In [14]:
# create full rank dataframe
df_fullRank = df.copy()

# booleans
df_fullRank["FinancialAssistance"] = df_fullRank["FinancialAssistance"].astype(int)
df_fullRank["Hypertension"] = df_fullRank["Hypertension"].astype(int)
df_fullRank["Diabetes"] = df_fullRank["Diabetes"].astype(int)
df_fullRank["Alcoholism"] = df_fullRank["Alcoholism"].astype(int)
df_fullRank["SMS_received"] = df_fullRank["SMS_received"].astype(int)
df_fullRank["NoShow"] = df_fullRank["NoShow"].astype(int)

# create dummies
genderDummies = pd.get_dummies(df["Gender"], dtype=int)
nhoodDummies = pd.get_dummies(df["Neighborhood"], dtype=int)
schedulingDayDummies = pd.get_dummies(
    df["SchedulingDate_dayName"], prefix="sch", dtype=int
)
appointDayDummies = pd.get_dummies(
    df["AppointmentDate_dayName"], prefix="apt", dtype=int
)
df_fullRank = df_fullRank.join(
    [genderDummies, nhoodDummies, schedulingDayDummies, appointDayDummies]
)

df_fullRank["intercept"] = 1

In [30]:
def significant_predictors(results, alpha=0.5):
    p_values = results.pvalues

    return f"significant predictors:\n{p_values[p_values < alpha].sort_values()}"


In [15]:
# prune duplicate patientIds
allPatients = df_fullRank.drop_duplicates(subset=["PatientId"])

In [31]:
# logistic regression for Patient demographic factors
# male omitted as baseline for Gender dummies
demographic_model = sm.Logit(allPatients["NoShow"], allPatients[[
    "intercept",
    "FinancialAssistance",
    "Hypertension",
    "Diabetes",
    "Alcoholism",
    "female"
    ]])
demographic_results = demographic_model.fit()

print(demographic_results.summary2())
print(significant_predictors(demographic_results))

Optimization terminated successfully.
         Current function value: 0.493242
         Iterations 6
                           Results: Logit
Model:               Logit             Method:            MLE       
Dependent Variable:  NoShow            Pseudo R-squared:  0.002     
Date:                2026-08-27 15:20  AIC:               61467.9484
No. Observations:    62298             BIC:               61522.1865
Df Model:            5                 Log-Likelihood:    -30728.   
Df Residuals:        62292             LL-Null:           -30799.   
Converged:           1.0000            LLR p-value:       4.0714e-29
No. Iterations:      6.0000            Scale:             1.0000    
--------------------------------------------------------------------
                     Coef.  Std.Err.    z     P>|z|   [0.025  0.975]
--------------------------------------------------------------------
intercept           -1.3907   0.0177 -78.7877 0.0000 -1.4253 -1.3561
FinancialAssistance  0.1890 

In [32]:
# logistic regression for neighborhood (all Patients)

# NoShow proportion per neighborhood
# 0 or 100% will be excluded from logistic regressions
nhoods = allPatients["Neighborhood"].unique()
nhood_NoShow = pd.DataFrame({"nhood": nhoods, "pNoShow": None})
nhood_NoShow.set_index("nhood", inplace=True)
nhoodsToIgnore = set()

for nhood in nhood_NoShow.index:
    nhood_appointments = allPatients[allPatients["Neighborhood"] == nhood]
    num_appointments = nhood_appointments.shape[0]
    num_NoShows = nhood_appointments["NoShow"].sum()

    pNoShow = num_NoShows / num_appointments

    if pNoShow == 0 or pNoShow == 1:
        nhoodsToIgnore.add(nhood)
        print(f"{nhood} will be ignored (pNoShow = {pNoShow:.2%})")
        pNoShow = None

    nhood_NoShow.loc[nhood, "pNoShow"] = pNoShow

nhood_NoShow.dropna(inplace=True)

# remove rows with visits to ignored neighborhoods
allPatients_clean = allPatients[
    ~allPatients["Neighborhood"].isin(nhoodsToIgnore)
].copy()

included_neighborhoods = nhood_NoShow.index.to_list()
baseline_neighborhood = included_neighborhoods[0]
predictor_neighborhoods = included_neighborhoods[1:]
print(f"baseline neighborhood: {baseline_neighborhood}")

X = allPatients_clean[["intercept"] + predictor_neighborhoods].astype(float)
y = allPatients_clean["NoShow"].astype(float)

patient_nhood_model = sm.Logit(y, X)
patient_nhood_model_results = patient_nhood_model.fit()

print(patient_nhood_model_results.summary2())
print(significant_predictors(patient_nhood_model_results))

ILHAS OCEÂNICAS DE TRINDADE will be ignored (pNoShow = 100.00%)
PARQUE INDUSTRIAL will be ignored (pNoShow = 0.00%)
baseline neighborhood: JARDIM DA PENHA
Optimization terminated successfully.
         Current function value: 0.491399
         Iterations 7
                           Results: Logit
Model:               Logit             Method:            MLE       
Dependent Variable:  NoShow            Pseudo R-squared:  0.006     
Date:                2026-08-27 15:20  AIC:               61381.4113
No. Observations:    62295             BIC:               62095.5426
Df Model:            78                Log-Likelihood:    -30612.   
Df Residuals:        62216             LL-Null:           -30796.   
Converged:           1.0000            LLR p-value:       2.6939e-39
No. Iterations:      7.0000            Scale:             1.0000    
--------------------------------------------------------------------
                     Coef.  Std.Err.    z     P>|z|   [0.025  0.975]
-----------

In [ ]:
# graphing helper functions
# def patientDistribution(patients: pd.DataFrame) -> dict[str, list[float]]:
#     numUniquePatients = patients.shape[0]
#     numFemale = patients.query("Gender == 'female'").shape[0]
#     numMale = patients.query("Gender == 'male'").shape[0]
#     numFinAssist = patients.query("FinancialAssistance == True").shape[0]
#     numHypertension = patients.query("Hypertension == True").shape[0]
#     numDiabetes = patients.query("Diabetes == True").shape[0]
#     numAlcoholism = patients.query("Alcoholism == True").shape[0]

#     return {
#         "All Patients": [numUniquePatients/allPatients.shape[0]],
#         "Female": [numFemale/numUniquePatients],
#         "Male": [numMale/numUniquePatients],
#         "FinancialAssistance": [numFinAssist/numUniquePatients],
#         "Hypertension": [numHypertension/numUniquePatients],
#         "Diabetes": [numDiabetes/numUniquePatients],
#         "Alcoholism": [numAlcoholism/numUniquePatients]
#     }

# def graphDistributions(data: dict[str, list[float]], title: str) -> tuple[plt.Figure, plt.Axes]:
#     fig, ax = plt.subplots()
#     ax.set_title(title)
#     ax.set_xlim(left=0, right=1)
#     ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0, decimals=0))

#     for label, numPatients in data.items():
#         p = ax.barh(label, numPatients, 0.8)
#         ax.bar_label(p, fmt=lambda x: f"{x * 100:.1f}%", label_type="edge", padding=3)

#     return [fig, ax]

In [ ]:
# allPatientDemo = patientDistribution(allPatients)
# showPatientDemo = patientDistribution(showPatients)
# noShowPatientDemo = patientDistribution(noShowPatients)

In [ ]:
# groupNames = (
#     "All Patients",
#     "Female",
#     "Male",
#     "FinancialAssistance",
#     "Hypertension",
#     "Diabetes",
#     "Alcoholism",
# )

# sources = {"All": allPatientDemo, "No Show": noShowPatientDemo, "Show": showPatientDemo}

# demographicDistributions = {
#     label: tuple(demo[k][0] for k in groupNames) for label, demo in sources.items()
# }

# fig, ax = plt.subplots(figsize=(7, 10))
# barContainers = ax.grouped_bar(
#     demographicDistributions,
#     tick_labels=groupNames,
#     group_spacing=1,
#     orientation="horizontal"
# )

# for container in barContainers.bar_containers:
#     ax.bar_label(
#         container, fmt=lambda x: f"{x * 100:.1f}%", label_type="edge", padding=3
#     )

# ax.set_xlabel("% of all Patients")
# ax.legend(loc="upper right", ncols=1)
# ax.set_xlim(0, 1)
# ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0, decimals=0))
# ax.tick_params(labelsize="small")
# ax.tick_params("y", labelrotation=90, labelrotation_mode="ytick")

In [ ]:
# Age, NumHandicaps and Neighborhood distributions

# Age histogram - try violinplot?
# dfAgeAll = allPatients["Age"]
# dfAgeShow = showPatients["Age"]
# dfAgeNoShow = noShowPatients["Age"]

# print(allPatients["Age"].max())

# maxAge = allPatients["Age"].max()
# bins = maxAge + 1
# range = (0, maxAge + 1)

# figAge, ((axAgeAll, axAgeShow, axAgeNoShow)) = plt.subplots(
#     nrows=3, ncols=1, figsize=(7, 9), sharey=True
# )

# axAgeAll.hist(x=dfAgeAll, histtype="bar", bins=bins, range=range, density=True)
# axAgeAll.set_title("Age Distribution: All Patients")
# axAgeAll.tick_params(labelsize="small")
# axAgeAll.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0, decimals=0))

# axAgeShow.hist(x=dfAgeShow, histtype="bar", bins=bins, range=range, density=True)
# axAgeShow.set_title("Age Distribution: Show Patients")
# axAgeShow.tick_params(labelsize="small")

# axAgeNoShow.hist(
#     x=dfAgeNoShow, histtype="bar", bins=bins, range=range, density=True
# )
# axAgeNoShow.set_title("Age Distribution: No Show Patients")
# axAgeNoShow.tick_params(labelsize="small")

In [ ]:
# NumHandicaps histogram
# dfHandcapAll = allPatients["NumHandicaps"]
# dfHandcapShow = showPatients["NumHandicaps"]
# dfHandcapNoShow = noShowPatients["NumHandicaps"]

# print(dfHandcapAll.value_counts())

# maxNumHandcaps = dfHandcapAll.max()
# bins = np.arange(0.5, maxNumHandcaps + 1.5, 1)
# range = (0, maxNumHandcaps + 1.5)

# figNumHandicaps, ((axHandcapAll, axHandcapShow, axHandcapNoShow)) = plt.subplots(
#     nrows=3, ncols=1, figsize=(7, 9), sharey=True, sharex=True
# )

# countsAll, binsAll, patchesAll = axHandcapAll.hist(
#     x=dfHandcapAll, histtype="bar", bins=bins, range=range, density=True, rwidth=0.8
# )
# axHandcapAll.set_title("NumHandicaps Distribution: All Patients")
# axHandcapAll.tick_params(labelsize="small")
# axHandcapAll.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0, decimals=0))
# axHandcapAll.bar_label(patchesAll, fmt=lambda x: f"{x * 100:.1f}%", label_type="edge")
# axHandcapAll.margins(y=0.1)

# countsShow, binsShow, patchesShow = axHandcapShow.hist(
#     x=dfHandcapShow, histtype="bar", bins=bins, range=range, density=True, rwidth=0.8
# )
# axHandcapShow.set_title("NumHandicaps Distribution: Show Patients")
# axHandcapShow.tick_params(labelsize="small")
# axHandcapShow.bar_label(patchesShow, fmt=lambda x: f"{x * 100:.1f}%", label_type="edge")
# axHandcapShow.margins(y=0.1)

# countsNoShow, binsNoShow, patchesNoShow = axHandcapNoShow.hist(
#     x=dfHandcapNoShow, histtype="bar", bins=bins, range=range, density=True, rwidth=0.8
# )
# axHandcapNoShow.set_title("NumHandicaps Distribution: No Show Patients")
# axHandcapNoShow.tick_params(labelsize="small")
# axHandcapNoShow.bar_label(
#     patchesNoShow, fmt=lambda x: f"{x * 100:.1f}%", label_type="edge"
# )
# axHandcapNoShow.margins(0.1)

### Do any factors related to the appointment correlate with a higher likelihood of a missed appointment?

#### Appointment factors
- `SchedulingDate` - day of week
- `AppointmentDate` - day of week
- number of days between `SchedulingDate` and `AppointmentDate`
- `SMS_received`
- number of appointments per `PatientId`
- has patient previously missed an appointment?

In [33]:
# logistic regression: schedule days
# baseline for SchedulingDate_dayName dummies: Saturday (Sunday not present in data)
schedule_day_model = sm.Logit(df_fullRank["NoShow"], df_fullRank[[
    "intercept",
    "sch_Monday",
    "sch_Tuesday",
    "sch_Wednesday",
    "sch_Thursday",
    "sch_Friday"]]
)
schedule_day_results = schedule_day_model.fit()

print(schedule_day_results.summary())
print(significant_predictors(schedule_day_results))

Optimization terminated successfully.
         Current function value: 0.502975
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:                 NoShow   No. Observations:               110521
Model:                          Logit   Df Residuals:                   110515
Method:                           MLE   Df Model:                            5
Date:                Thu, 27 Aug 2026   Pseudo R-squ.:               9.404e-05
Time:                        15:22:29   Log-Likelihood:                -55589.
converged:                       True   LL-Null:                       -55595.
Covariance Type:            nonrobust   LLR p-value:                   0.06329
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
intercept        -3.1355      1.022     -3.069      0.002      -5.138      -1.133
sch_Monday        1.

In [34]:
# logistic regression: appointment days
# baseline for AppointmentDate_dayName: Saturday (Sunday not present in data)
appointment_day_model = sm.Logit(df_fullRank["NoShow"], df_fullRank[[
    "intercept",
    "apt_Monday",
    "apt_Tuesday",
    "apt_Wednesday",
    "apt_Thursday",
    "apt_Friday"
]])
appointment_day_results = appointment_day_model.fit()

print(appointment_day_results.summary())
print(significant_predictors(appointment_day_results))

Optimization terminated successfully.
         Current function value: 0.502898
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:                 NoShow   No. Observations:               110521
Model:                          Logit   Df Residuals:                   110515
Method:                           MLE   Df Model:                            5
Date:                Thu, 27 Aug 2026   Pseudo R-squ.:               0.0002475
Time:                        15:24:01   Log-Likelihood:                -55581.
converged:                       True   LL-Null:                       -55595.
Covariance Type:            nonrobust   LLR p-value:                 4.521e-05
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
intercept        -1.2040      0.380     -3.168      0.002      -1.949      -0.459
apt_Monday       -0.

In [36]:
# logistic regression for numDaysCreatedToScheduled
numDays_model = sm.Logit(df_fullRank["NoShow"], df_fullRank[["intercept", "numDaysCreatedToScheduled"]])
numDays_results = numDays_model.fit()


print(numDays_results.summary())
print(significant_predictors(numDays_results))

Optimization terminated successfully.
         Current function value: 0.487885
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:                 NoShow   No. Observations:               110521
Model:                          Logit   Df Residuals:                   110519
Method:                           MLE   Df Model:                            1
Date:                Thu, 27 Aug 2026   Pseudo R-squ.:                 0.03009
Time:                        15:25:15   Log-Likelihood:                -53922.
converged:                       True   LL-Null:                       -55595.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
intercept                    -1.6779      0.010   -175.079      0.000      -1.

In [37]:
# SMS_Recieved
sms_model = sm.Logit(df_fullRank["NoShow"], df_fullRank[["intercept", "SMS_received"]])
sms_results = sms_model.fit()

print(sms_results.summary())
print(significant_predictors(sms_results))

Optimization terminated successfully.
         Current function value: 0.495313
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:                 NoShow   No. Observations:               110521
Model:                          Logit   Df Residuals:                   110519
Method:                           MLE   Df Model:                            1
Date:                Thu, 27 Aug 2026   Pseudo R-squ.:                 0.01533
Time:                        15:25:50   Log-Likelihood:                -54742.
converged:                       True   LL-Null:                       -55595.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
intercept       -1.6072      0.010   -164.198      0.000      -1.626      -1.588
SMS_received     0.6415

In [ ]:
# number of appointments per patient

In [ ]:
# has patient previously missed an appointment

In [ ]:
# Continue to explore the data to address your additional research
#   questions. Add more headers as needed if you have more questions to
#   investigate.

<a id='conclusions'></a>
## Conclusions

> **Tip**: Finally, summarize your findings and the results that have been performed in relation to the question(s) provided at the beginning of the analysis. Summarize the results accurately, and point out where additional research can be done or where additional information could be useful.

> **Tip**: Make sure that you are clear with regards to the limitations of your exploration. You should have at least 1 limitation explained clearly. 

> **Tip**: If you haven't done any statistical tests, do not imply any statistical conclusions. And make sure you avoid implying causation from correlation!

> **Tip**: Once you are satisfied with your work here, check over your report to make sure that it is satisfies all the areas of the rubric (found on the project submission page at the end of the lesson). You should also probably remove all of the "Tips" like this one so that the presentation is as polished as possible.

## Submitting your Project 

> **Tip**: Before you submit your project, you need to create a .html or .pdf version of this notebook in the workspace here. To do that, run the code cell below. If it worked correctly, you should see output that starts with `NbConvertApp] Converting notebook`, and you should see the generated .html file in the workspace directory (click on the orange Jupyter icon in the upper left).

> **Tip**: Alternatively, you can download this report as .html via the **File** > **Download as** submenu, and then manually upload it into the workspace directory by clicking on the orange Jupyter icon in the upper left, then using the Upload button.

> **Tip**: Once you've done this, you can submit your project by clicking on the "Submit Project" button in the lower right here. This will create and submit a zip file with this .ipynb doc and the .html or .pdf version you created. Congratulations!

In [ ]:
# Running this cell will execute a bash command to convert this notebook to an .html file
!python -m nbconvert --to html Investigate_a_Dataset.ipynb